In [7]:
import sys, io, json
sys.path.insert(0, "../..")
import helpers

import boto3
import pandas as pd
from moto import mock_aws

In [8]:
with mock_aws():
    # boto3.client(
    # "s3", 
    # region_name="us-east-1", 
    # aws_access_key_id="testing", 
    # aws_secret_access_key="testing")
    s3 = helpers.make_boto3_client("s3")

    s3.create_bucket(Bucket="my-datalake-use1")

    # ANy other region: must pass CreateBucketConfiguration
    s3.create_bucket(
        Bucket="my-datalake-eu",
        CreateBucketConfiguration={"LocationConstraint": "eu-west-1"},
    )

    buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]
    print("Buckets created", buckets)

Buckets created ['my-datalake-use1', 'my-datalake-eu']


Add tags (cost allocation tags are important in real DE environments)

In [9]:
with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="tagged-lake")
    s3.put_bucket_tagging(
        Bucket="tagged-lake",
        Tagging={"TagSet": [
            {"Key": "Team", "Value": "DataEngineering"},
            {"Key": "CostCenter", "Value": "DE-001"},
        ]},
    )
    tags = s3.get_bucket_tagging(Bucket="tagged-lake")["TagSet"]
    print("Tags", tags)

Tags [{'Key': 'Team', 'Value': 'DataEngineering'}, {'Key': 'CostCenter', 'Value': 'DE-001'}]


Bucket Locations

In [10]:
with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="location-demo")
    loc = s3.get_bucket_location(Bucket="location-demo")
    # us-east-1 returns None for LocationConstraint
    print("Location:", loc["LocationConstraint"])

Location: None


Object Operations

In [13]:
with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="objects-demo")
    
    # Upload raw objects
    s3.put_object(Bucket="objects-demo", Key="raw/hello.txt", Body=b"Hello, Data Lake!")

    # Upload a string (must encode to bytes)
    s3.put_object(Bucket="objects-demo", Key="raw/data.csv", Body="id, name\n1, Alice\n2,Bob".encode())

    response = s3.get_object(Bucket="objects-demo", Key="raw/hello.txt")
    text = response["Body"].read().decode()
    print("Downloaded:", text)

Downloaded: Hello, Data Lake!


Upload & Download Parquet

In [14]:
with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="parquet-demo")

    df = pd.DataFrame({"order_id": range(5), "amount": [10.5, 20.0, 15.3, 8.9, 32.1]})
    helpers.upload_df_as_parquet(s3, df, "parquet-demo", "order/2024-01.parquet")

    df_back = helpers.download_df_from_parquet(s3, "parquet-demo", "order/2024-01.parquet")
    print(df_back)
    assert df_back.shape == (5, 2), "Shape mismatch!"
    print("Parquet round-trip: OK")

   order_id  amount
0         0    10.5
1         1    20.0
2         2    15.3
3         3     8.9
4         4    32.1
Parquet round-trip: OK
